#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [2]:

MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


#`Setup And Update ComfyUI`



In [6]:
from pathlib import Path
import os

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
COMFYUI_LAUNCH_ARGS = "--dont-print-server --disable-auto-launch"  #@param {type:"string"}
WORKSPACE = '/root/comfy/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

os.environ["PIP_CACHE_DIR"] = "/content/pip-cache"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

if DRIVE_PATH:

    %cd {DRIVE_PATH}

os.environ["WORKSPACE"] = WORKSPACE

if not Path(WORKSPACE).exists():
  !echo -= Initial setup ComfyUI =-
  !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
%cd {WORKSPACE}

# Update ComfyUI repo if requested
if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git -C {WORKSPACE} pull

!echo -= Install dependencies =-
!pip install -q --upgrade pip

%cd {WORKSPACE}

# --- ComfyUI Manager: install using comfy-cli ---
import subprocess
import sys

print('Installing comfy-cli and running comfy install (may prompt interactively)')
try:
  subprocess.run([sys.executable, '-m', 'pip', 'install', 'comfy-cli'], check=True)
except Exception as e:
  print('Failed to install comfy-cli:', e)
  raise
# Run comfy install; this may prompt for choices. Run manually if you need to customise options.
try:
  subprocess.run(['comfy', 'install'], check=True)
except Exception as e:
  print('comfy install returned an error (may be interactive or require manual run):', e)
  print('You can run `comfy install` yourself in the notebook or choose another installation method')


/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
/content/ComfyUI
Installing comfy-cli and running comfy install (may prompt interactively)
comfy install returned an error (may be interactive or require manual run): Command '['comfy', 'install']' returned non-zero exit status 1.
You can run `comfy install` yourself in the notebook or choose another installation method


# `Models Download`

## Download MODELS

In [8]:
!pip install -q -U "huggingface_hub[cli]" hf_transfer

import os
import subprocess
import getpass
from pathlib import Path

os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

# Prefer environment variables, otherwise prompt the user securely for the token
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass.getpass("Enter your Hugging Face token (it will be hidden): ")
    except Exception:
        HF_TOKEN = input("Enter your Hugging Face token: ")
if not HF_TOKEN:
    raise ValueError("Hugging Face token not provided. Set HF_TOKEN env var or provide it when prompted.")
print("Loaded Hugging Face token: yes")

MODEL_DIRS = {
    "diffusion": Path(WORKSPACE) / "models" / "diffusion_models",
    "text_encoder": Path(WORKSPACE) / "models" / "text_encoders",
    "vae": Path(WORKSPACE) / "models" / "vae",
}
for model_dir in MODEL_DIRS.values():
    model_dir.mkdir(parents=True, exist_ok=True)
(Path(WORKSPACE) / "custom_nodes").mkdir(parents=True, exist_ok=True)


def hf_download(repo_id, filename, local_dir, target_name=None):
    local_dir = Path(local_dir)
    target = local_dir / (target_name or Path(filename).name)
    if target.exists() and target.stat().st_size > 0:
        print(f"Already present: {target}")
        return target

    cmd = [
        "hf",
        "download",
        repo_id,
        filename,
        "--local-dir",
        str(local_dir),
        "--token",
        HF_TOKEN,
    ]
    env = os.environ.copy()
    result = subprocess.run(cmd, env=env, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"hf download failed for {repo_id}/{filename}")

    downloaded = local_dir / filename
    if target_name and downloaded.exists() and downloaded != target:
        target.parent.mkdir(parents=True, exist_ok=True)
        downloaded.replace(target)
        parent = downloaded.parent
        while parent != local_dir:
            try:
                parent.rmdir()
            except OSError:
                break
            parent = parent.parent
    print(f"Ready: {target}")
    return target

hf_download(
    "black-forest-labs/FLUX.2-klein-9b-fp8",
    "flux-2-klein-9b-fp8.safetensors",
    MODEL_DIRS["diffusion"],
)
hf_download(
    "ponpoke/flux2-klein-9b-uncensored-text-encoder",
    "flux2-klein-9b-uncensored-q8_0.gguf",
    MODEL_DIRS["text_encoder"],
)
hf_download(
    "black-forest-labs/FLUX.2-klein-9B",
    "vae/diffusion_pytorch_model.safetensors",
    MODEL_DIRS["vae"],
    target_name="FLUX.2-klein-9B-vae.safetensors",
)


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
Loaded Hugging Face token: yes
Ready: /content/ComfyUI/models/diffusion_models/flux-2-klein-9b-fp8.safetensors
Ready: /content/ComfyUI/models/text_encoders/flux2-klein-9b-uncensored-q8_0.gguf
Ready: /content/ComfyUI/models/vae/FLUX.2-klein-9B-vae.safetensors


PosixPath('/content/ComfyUI/models/vae/FLUX.2-klein-9B-vae.safetensors')

## INSTALL CUSTOM NODES


In [11]:
from collections import OrderedDict
from pathlib import Path
import subprocess
import sys

COMFYUI_PATH = Path("/root/comfy/ComfyUI")
CUSTOM_NODES_PATH = COMFYUI_PATH / "custom_nodes"
CUSTOM_NODES_PATH.mkdir(parents=True, exist_ok=True)

custom_node_repos = OrderedDict((
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF.git"),
    ("ComfyUI-Easy-Use", "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-Crystools", "https://github.com/crystian/ComfyUI-Crystools.git"),
    ("ComfyUI-Lora-Manager", "https://github.com/jthickma/ComfyUI-Lora-Manager.git"),
))

for name, url in custom_node_repos.items():
    node_path = CUSTOM_NODES_PATH / name
    if node_path.exists():
        print(f"Updating {name}")
        try:
            subprocess.run(["git", "-C", str(node_path), "pull", "--ff-only"], check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            print(f"Failed to update {name}: {e.stderr}")
            print('You can try updating manually inside the node folder')
    else:
        print(f"Cloning {name}")
        try:
            # try a shallow clone first to save time and bandwidth
            subprocess.run(["git", "clone", "--depth", "1", url, str(node_path)], check=True, capture_output=True, text=True)
        except subprocess.CalledProcessError as e:
            print(f"Shallow clone failed for {name}: {e.stderr}")
            print("Retrying full clone without --depth...")
            try:
                subprocess.run(["git", "clone", url, str(node_path)], check=True, capture_output=True, text=True)
            except subprocess.CalledProcessError as e2:
                print(f"Clone failed for {name}: {e2.stderr}")
                print(f"Skipping {name} due to clone errors. If this repo is private or rate-limited, provide credentials or try later.")
                continue

    requirements = node_path / "requirements.txt"
    if requirements.exists():
        print(f"Installing requirements for {name}")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Failed to install requirements for {name}: {e}")
            print("You can install them manually later by running: pip install -r requirements.txt")
    else:
        print(f"No requirements.txt for {name}")

%cd {WORKSPACE}


Cloning rgthree-comfy
Installing requirements for rgthree-comfy
Cloning ComfyUI-KJNodes
Installing requirements for ComfyUI-KJNodes
Cloning ComfyUI-GGUF
Installing requirements for ComfyUI-GGUF
Cloning ComfyUI-Easy-Use
Installing requirements for ComfyUI-Easy-Use
Cloning ComfyUI-Crystools
Installing requirements for ComfyUI-Crystools
Cloning ComfyUI-Lora-Manager
Installing requirements for ComfyUI-Lora-Manager
/root/comfy/ComfyUI


## COLAB OPTIMIZATIONS


In [ ]:
# Colab optimization notes
# - Keep DRIVE_PATH empty for fastest ephemeral installs; set it only when you need model persistence.
# - HF transfers are accelerated with hf_transfer and cached under /content/hf-cache to avoid slow Drive metadata churn.
# - COMFYUI_LAUNCH_ARGS defaults to --lowvram for Colab GPUs; remove it on high-VRAM runtimes if you want more speed.
# - The custom-node cell installs ComfyUI-GGUF for workflows that load GGUF text encoders.


## LIST MODELS

In [ ]:
%cd {WORKSPACE}
!find ./models/diffusion_models ./models/text_encoders ./models/vae -maxdepth 2 -type f -print -exec ls -lh {} \;


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [12]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-06-06 05:04:43--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64.deb [following]
--2026-06-06 05:04:44--  https://github.com/cloudflare/cloudflared/releases/download/2026.5.2/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/0ec93a74-a3bc-474a-bd4b-e852ababcbb3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-06T05%3A42%3A13Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
%cd /root/comfy/ComfyUI
!python main.py {COMFYUI_LAUNCH_ARGS}


# `START ComfyUI & Expose Server`

## Download Prerequisits

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## CF Tunnel

In [15]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /root/comfy/ComfyUI
!python main.py {COMFYUI_LAUNCH_ARGS}


/root/comfy/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['apply_rope', 'apply_rope1', 'apply_rope_split_half', 'apply_rope_split_half1', 'dequantize_mxfp8', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 

## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd {WORKSPACE}
!python main.py {COMFYUI_LAUNCH_ARGS}
